# QMCPy — Acceptance-Rejection True Measure
## Based on Zhu & Dick (2014): *Discrepancy bounds for deterministic acceptance-rejection samplers*

---

### What this notebook does

This notebook introduces a new `AcceptanceRejection` **TrueMeasure** class for the **QMCPy** library.  It follows the QMCPy `AbstractTrueMeasure` interface so users can swap it into any existing QMCPy workflow where they currently use `Uniform` or `Gaussian`.

The class is built directly from the three algorithms in Zhu & Dick (2014):

| Algorithm | Class | Key idea |
|-----------|-------|----------|
| **Alg. 2** — DAR | `AcceptanceRejection` | Filter a (t,m,s)-net through `L·u ≤ ψ(x)` |
| **Alg. 3** — DAR-Real | `AcceptanceRejectionReal` | Same, after inverse Rosenblatt transform to ℝ^d |
| **Alg. 4** — DRAR | `ReducedAcceptanceRejection` | Hybrid: inversion where possible, A-R where needed |

**Why QMC instead of random A-R?**  
Standard (random) acceptance-rejection gives i.i.d. samples with star discrepancy converging at the Monte Carlo rate O(N^{-1/2}).  Replacing the driver with a genuine **(t,m,s)-net** (Sobol, Halton, …) gives a discrepancy of order **O(N^{-1/s})** — and empirically much better — per Theorem 1 of Zhu & Dick.

> **Critical requirement (M = 2^m):** The number of driver points *must* be a power of the base (here 2) so that the points form a proper (t,m,s)-net and the discrepancy bound holds.  The class enforces this automatically.


## Cell 1 — Imports

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import qmc as scipy_qmc
from scipy.stats import norm, expon
from scipy.integrate import quad
import warnings
warnings.filterwarnings('ignore')

# ── local module (will live in qmcpy/true_measure/acceptance_rejection.py) ──
import sys, os
sys.path.insert(0, '.')   # make acceptance_rejection.py importable
from acceptance_rejection import (
    AcceptanceRejection,
    AcceptanceRejectionReal,
    ReducedAcceptanceRejection,
)

plt.rcParams.update({'font.size': 12, 'figure.dpi': 110})
print("All imports successful.")
print(f"scipy version: {__import__('scipy').__version__}")
print()
print("Classes available:")
print("  AcceptanceRejection         (Algorithm 2 — DAR on [0,1]^d)")
print("  AcceptanceRejectionReal     (Algorithm 3 — DAR on ℝ^d)")
print("  ReducedAcceptanceRejection  (Algorithm 4 — DRAR)")


All imports successful.
scipy version: 1.14.1

Classes available:
  AcceptanceRejection         (Algorithm 2 — DAR on [0,1]^d)
  AcceptanceRejectionReal     (Algorithm 3 — DAR on ℝ^d)
  ReducedAcceptanceRejection  (Algorithm 4 — DRAR)


## Cell 2 — The driver: QMC vs Random

The **entire point of Zhu & Dick (2014)** is that the driver sequence must be a genuine low-discrepancy sequence, **not `np.random.rand`**.

We define two driver classes that share the same interface:
- `SobolDriver` — wraps `scipy.stats.qmc.Sobol`, always generates exactly M = 2^m points (required for the (t,m,s)-net guarantee)
- `RandomDriver` — baseline, equivalent to the standard random A-R used as comparison in the paper
